In [ ]:
#| hide
#| default_exp nbio

# nbio

> Reading and writing Jupyter notebooks

In [ ]:
#| export
from fastcore.basics import *
from fastcore.xtras import rtoken_hex,clean_cli_output,take_lines,str_diff
from fastcore.imports import *
from fastcore.ansi import ansi2html
from fastcore.meta import delegates,splice_sig
from fastcore.tools import insert_line,str_replace,strs_replace,replace_lines,del_lines,lnhash

import ast,functools
from collections import defaultdict
from pprint import pformat,pprint
from json import loads,dumps

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq,expect_fail

## Reading a notebook

A notebook is just a json file.

In [ ]:
#| exports
def _read_json(self, encoding=None, errors=None):
    return loads(Path(self).read_text(encoding=encoding, errors=errors))

In [ ]:
minimal_fn = Path('../tests/minimal.ipynb')
minimal_txt = AttrDict(_read_json(minimal_fn))

It contains two sections, the `metadata`...:

In [ ]:
minimal_txt.metadata

{'solveit_dialog_mode': 'learning', 'solveit_ver': 2}

...and, more importantly, the `cells`:

In [ ]:
minimal_txt.cells

[{'cell_type': 'markdown',
  'id': '801558df',
  'metadata': {},
  'source': ['## A minimal notebook']},
 {'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00'},
  'outputs': [{'data': {'text/plain': ['2']},
    'execution_count': 0,
    'metadata': {},
    'output_type': 'execute_result'}],
  'source': ['# Do some arithmetic\n', '1+1']}]

The second cell here is a `code` cell, however it contains no outputs, because it hasn't been executed yet. To execute a notebook, we first need to convert it into a format suitable for `nbclient` (which expects some `dict` keys to be available as attrs, and some available as regular `dict` keys). Normally, `nbformat` is used for this step, but it's rather slow and inflexible, so we'll write our own function based on `fastcore`'s handy `dict2obj`, which makes all keys available as both attrs *and* keys.

In [ ]:
#| export
_non_text_split_mimes = {'application/javascript', 'image/svg+xml'}

def _is_json_mime(mime): return mime=='application/json' or (mime.startswith('application/') and mime.endswith('+json'))

def _rejoin_mime(data):
    "Join list-of-lines values in mimebundle `data` in place, like nbformat's `rejoin_lines`"
    for k,v in data.items():
        if not _is_json_mime(k) and isinstance(v,list) and all(isinstance(o,str) for o in v): data[k] = ''.join(v)

def _rejoin_cell(c):
    "Join list-of-lines text in `c`'s attachments and outputs in place (`source` is handled by `set_source`)"
    for att in c.get('attachments', {}).values(): _rejoin_mime(att)
    if c.get('cell_type')=='code':
        for o in c.get('outputs', []):
            if o.get('output_type') in ('execute_result','display_data'): _rejoin_mime(o.get('data', {}))
            elif o.get('output_type')=='stream' and isinstance(o.get('text'), list): o['text'] = ''.join(o['text'])

In [ ]:
#| export
# from https://github.com/quarto-dev/quarto-cli/blob/main/src/resources/jupyter/notebook.py
langs = defaultdict(
    lambda: '#',  r = "#", python = "#", julia = "#", scala = "//", matlab = "%", csharp = "//", fsharp = "//",
    c = ["/*","*/"], css = ["/*","*/"], sas = ["*",";"], powershell = "#", bash = "#", sql = "--", mysql = "--", psql = "--",
    lua = "--", cpp = "//", cc = "//", stan = "#", octave = "#", fortran = "!", fortran95 = "!", awk = "#", gawk = "#", stata = "*",
    java = "//", groovy = "//", sed = "#", perl = "#", ruby = "#", tikz = "%", javascript = "//", js = "//", d3 = "//", node = "//",
    sass = "//", coffee = "#", go = "//", asy = "//", haskell = "--", dot = "//", apl = "⍝")

def nb_lang(nb): return nested_attr(nb, 'metadata.kernelspec.language', 'python')

Each notebook language has its own comment character(s), taken from Quarto's table; `nb_lang` reads the notebook's language from its kernelspec, defaulting to Python. Cells record their language in `lang_` (a per-cell `metadata.language` overrides the notebook default), which the directive functions below use to recognize comments.

In [ ]:
#| export
class NbCell(AttrDict):
    def __init__(self, idx, cell, lang='python'):
        super().__init__(cell)
        self.idx_ = idx
        self.lang_ = nested_attr(self, 'metadata.language', lang)
        if 'id' not in self: self.id = rtoken_hex(4)
        if 'source' in self: self.set_source(self.source)
        _rejoin_cell(self)

    def __setattr__(self, k, v):
        prop = getattr(type(self), k, None)
        if isinstance(prop, property) and prop.fset: prop.fset(self, v)
        else: super().__setattr__(k, v)

    def __setitem__(self, k, v):
        super().__setitem__(k, v)
        if k in ('source','metadata'):
            for c in ('_parsed_','_directives_','_meta_names_'):
                if c in self: del(self[c])

    def set_source(self, source): self.source = ''.join(source)

    def parsed_(self):
        if self.cell_type!='code' or self.source.strip()[:1] in ['%', '!']: return
        if '_parsed_' not in self:
            try: self._parsed_ = ast.parse(self.source).body
            # you can assign the result of ! to a variable in a notebook cell
            # which will result in a syntax error if parsed with the ast module.
            except SyntaxError: return
        return self._parsed_

    def __hash__(self): return hash(self.id) if 'id' in self else hash(self.source) + hash(self.cell_type)
    def __eq__(self,o): return self.id==o.id if 'id' in self and 'id' in o else self.source==o.source and self.cell_type==o.cell_type

We use an `AttrDict` subclass which has some basic functionality for accessing notebook cells.

Two cells are equal if they have the same `id`, even when their content differs -- `id` is what identifies *which* cell this is, so this is what lets `list.index`/`list.remove` (used by `Notebook`'s cell-moving methods) find the right cell even when another cell elsewhere has identical source. Cells lacking an `id` fall back to comparing `source`/`cell_type`.

In [ ]:
c1 = NbCell(0, dict(cell_type='code', source='a', id='x'))
c2 = NbCell(0, dict(cell_type='code', source='b', id='x'))
c3 = NbCell(0, dict(cell_type='code', source='a', id='y'))
test_eq(c1, c2)      # same id, different content -- still equal
assert c1 != c3      # same content, different id -- not equal

cells = [c1, c3]  # duplicate content ('a'), different ids
test_eq(cells.index(c3), 1)  # finds the actual cell, not the first with matching content


In [ ]:
#| export
def _dict2obj(d, list_func=list, dict_func=AttrDict):
    "Convert (possibly nested) dicts (or lists of dicts) to `AttrDict`"
    if isinstance(d, list): return list(map(_dict2obj, d))
    if not isinstance(d, dict): return d
    return dict_func(**{k:_dict2obj(v) for k,v in d.items()})

def dict2nb(js=None, **kwargs):
    "Convert dict `js` to an `AttrDict`, "
    nb = _dict2obj(js or kwargs)
    nb.cells = [NbCell(i, o, nb_lang(nb)) for i,o in enumerate(nb.cells)]
    return nb

We can now convert our JSON into this `nbclient`-compatible format, which pretty prints the source code of cells in notebooks.

In [ ]:
minimal = dict2nb(minimal_txt)
cell = minimal.cells[1]
cell

<div class="prose" markdown="1">

```python
{ 'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'idx_': 1,
  'lang_': 'python',
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00'},
  'outputs': [ { 'data': {'text/plain': '2'},
                 'execution_count': 0,
                 'metadata': {},
                 'output_type': 'execute_result'}],
  'source': '# Do some arithmetic\n1+1'}
```

</div>

The abstract syntax tree of source code cells is available in the `parsed_` property:

In [ ]:
cell.parsed_(), cell.parsed_()[0].value.op

([1 + 1], )

In [ ]:
#| export
def read_nb(path):
    "Return notebook at `path`"
    res = dict2nb(_read_json(path, encoding='utf-8'))
    res['path_'] = str(path)
    return res

This reads the JSON for the file at `path` and converts it with `dict2nb`. For instance:

In [ ]:
minimal = read_nb(minimal_fn)
str(minimal.cells[0])

"{'cell_type': 'markdown', 'id': '801558df', 'metadata': {}, 'source': '## A minimal notebook', 'idx_': 0, 'lang_': 'python'}"

The file name read is stored in `path_`:

In [ ]:
minimal.path_

'../tests/minimal.ipynb'

## Creating a notebook

In [ ]:
#| export
def mk_cell(
    text,  # `source` attr in cell
    cell_type='code',  # `cell_type` attr in cell
    **kwargs # any other attrs to add to cell
):
    "Create an `NbCell` containing `text`"
    assert cell_type in {'code', 'markdown', 'raw'}
    if 'metadata' not in kwargs: kwargs['metadata']={}
    if cell_type == 'code':
        kwargs.setdefault('outputs', [])
        kwargs.setdefault('execution_count', None)
    return NbCell(0, dict(cell_type=cell_type, source=text, directives_={}, **kwargs))

In [ ]:
mk_cell('print(1)', execution_count=0)

<div class="prose" markdown="1">

```python
{ 'cell_type': 'code',
  'directives_': {},
  'execution_count': 0,
  'id': '37dd5d5e',
  'idx_': 0,
  'lang_': 'python',
  'metadata': {},
  'outputs': [],
  'source': 'print(1)'}
```

</div>

In [ ]:
#| export
def new_nb(cells=None, meta=None, nbformat=4, nbformat_minor=5):
    "Returns an empty new notebook"
    cells = [o if isinstance(o,dict) else mk_cell(o) for o in cells or []]
    return dict2nb(cells=cells or [],metadata=meta or {},nbformat=nbformat,nbformat_minor=nbformat_minor)

Use this function when creating a new notebook. Useful for when you don't want to create a notebook on disk first and then read it.

In [ ]:
test_eq(new_nb().cells, [])

## Directives

nbdev and Quarto put *directives* in comments at the top of a cell: lines like `#| export` or `#| eval: false`, ending at the first non-comment line. All spellings are equivalent: `#| foo: bar`, `#| foo:bar`, and `#| foo bar` parse to the same thing, and a directive's value is the raw text after its name, so nothing is lost to tokenization. A value of `true` means the same as no value at all: `#| hide`, `#| hide: true`, and `#| hide:` are one directive (which also means no directive can take the literal word `true` as its value). Directives can also be stored in cell metadata, as a dict under an `nbdev` key, with the same value domain: every value is a str, `\"true\"` meaning bare (non-str values raise, so a JSON `false` fails loudly rather than silently differing from `\"false\"`). When the same name appears in both places, the comment wins.

In [ ]:
#| export
def _dir_pre(lang=None): return fr"\s*{langs[lang]}\s*\|"
_cell_mgc = re.compile(r"^\s*%%\w+")

def first_code_ln(code_list, re_pattern=None, lang='python'):
    "get first line number where code occurs, where `code_list` is a list of code"
    if re_pattern is None: re_pattern = _dir_pre(lang)
    return first(i for i,o in enumerate(code_list) if o.strip() != '' and not re.match(re_pattern, o) and not _cell_mgc.match(o))

In [ ]:
_tst = """ 
#| default_exp
 #| export
#| hide_input
foo
"""
test_eq(first_code_ln(_tst.splitlines(True)), 4)

In [ ]:
#| hide

# test for cell magics
_tst = """%%timeit
#| hide
 #| export
foo
"""
test_eq(first_code_ln(_tst.splitlines(True)), 3)

# test when there is line magic
_tst = """
#| hide
%line_magic
 #| export
foo
"""
test_eq(first_code_ln(_tst.splitlines(True)),2)

In [ ]:
#| export
def _directive(s, lang='python'):
    "Parse a directive line into `(name, value)`; None if `s` is not a directive line"
    if not re.match(_dir_pre(lang), s): return None
    s = re.sub('^'+_dir_pre(lang), '', s).strip()
    m = re.match(r'([^\s:]+)\s*:?\s?(.*?)\s*$', s, flags=re.S)
    if not m: return None
    name,v = m.groups()
    return name, ('' if v=='true' else v)

`_directive` parses one directive line into its name and value. The value is the raw text after the name (and optional colon), byte-exact apart from one leading space and trailing whitespace, so both spellings parse identically; a value of `true` collapses to `''`, the same as a bare directive. Non-directive lines (including cell magics) parse to `None`.

In [ ]:
test_eq(_directive('#| export: utils'), ('export','utils'))
test_eq(_directive('#| export utils'),  ('export','utils'))
test_eq(_directive('#| export:utils'),  ('export','utils'))
test_eq(_directive('#| hide'), ('hide',''))
test_eq(_directive('#| hide:'), ('hide',''))
test_eq(_directive('#| hide: true'), ('hide',''))
test_eq(_directive(' # | woo:baz'), ('woo','baz'))
test_eq(_directive('#| fig-cap: "two  spaces: kept"'), ('fig-cap','"two  spaces: kept"'))
test_eq(_directive('#| filter_stream secret apikey'), ('filter_stream','secret apikey'))
test_eq(_directive('%%timeit'), None)
test_eq(_directive('# plain comment'), None)

In [ ]:
#| export
def _meta_directives(meta):
    "Directives from the `nbdev` dict in metadata dict `meta`, as `{name: value}`; values must be str (`'true'` for a bare directive)"
    d = (meta or {}).get('nbdev',{})
    if bad := [k for k,v in d.items() if not isinstance(v,str)]:
        raise TypeError(f"`nbdev` metadata directive values must be str (e.g 'true'/'false'), got non-str for: {bad}")
    return {k:'' if v=='true' else v for k,v in d.items()}

def _dir_attrs(meta):
    "Metadata directives as XML attrs: bare directives become True, so they render as bare attributes"
    return {k:v or True for k,v in _meta_directives(meta).items()}

def _unparse_dir(v):
    "Inverse of `_meta_directives` value parsing: value string back to a metadata value"
    return 'true' if v=='' else v

def _dir_line(k, v, lang='python', quarto=False):
    "A canonical directive comment line; with `quarto`, bare directives render as `: true`"
    if quarto and not v: v = 'true'
    return f"{langs[lang]}| {k}" + (f": {v}" if v else '') + '\n'

In [ ]:
#| export
@patch
def _partition(self:NbCell):
    "Split source into leading directive lines (with magics and blanks) and the rest"
    dirs = take_lines(self.source, fr'^({_dir_pre(self.lang_)}|\s*%%\w+|\s*$)')
    return dirs.splitlines(True), self.source[len(dirs):].splitlines(True)

def _directives_get(self):
    "Directives from leading comments and the metadata `nbdev` key; comments win. Assign a modified copy back to update the cell."
    if '_directives_' not in self:
        dirs,_ = self._partition()
        cmts = dict(t for t in (_directive(s, self.lang_) for s in dirs) if t)
        meta = _meta_directives(self.get('metadata'))
        self['_meta_names_'] = set(meta) - set(cmts)
        self['_directives_'] = cmts | {k:v for k,v in meta.items() if k not in cmts}
    return dict(self['_directives_'])

def _directives_set(self, d):
    if d == self.directives: return
    mnames = self['_meta_names_']
    meta = {k:_unparse_dir(v) for k,v in d.items() if k in mnames}
    lines = [_dir_line(k, v, self.lang_) for k,v in d.items() if k not in mnames]
    if meta: self.setdefault('metadata', AttrDict())['nbdev'] = meta
    elif 'nbdev' in self.get('metadata', {}): del self.metadata['nbdev']
    dirs,code = self._partition()
    self.set_source(''.join([o for o in dirs if _cell_mgc.match(o)] + lines + code))

NbCell.directives = property(_directives_get, _directives_set)

`directives` reads as a plain dict of name to value string: `{'export': 'utils', 'hide': ''}` (bare directives have value `''`). The getter returns a copy, so to edit, modify the copy and assign it back; the setter regenerates the comment block in canonical colon form (cell magics stay first, untouched). Directives that came from cell metadata are tracked by name and written back to metadata rather than to comments; names added through the setter become comments.

In [ ]:
c = mk_cell('#| export: utils\n#| hide\ndef f(): pass')
test_eq(c.directives, {'export':'utils', 'hide':''})

d = c.directives
d.pop('hide')
d['eval'] = 'false'
c.directives = d
test_eq(c.source, '#| export: utils\n#| eval: false\ndef f(): pass')

In [ ]:
#| hide
# any way of writing source invalidates the cache: set_source, attribute, or item assignment
c = mk_cell('#| hide\n1')
test_eq(c.directives, {'hide':''})
c.source = '1'
test_eq(c.directives, {})
c['source'] = '#| export\n1'
test_eq(c.directives, {'export':''})

In [ ]:
#| hide
# assigning metadata also invalidates (mutate a copy and assign back for nested edits)
c.metadata = dict(nbdev=dict(eval='false'))
test_eq(c.directives, {'export':'', 'eval':'false'})

Metadata directives merge in (comments win on conflict), and edits route back to where each directive came from. The setter writes bare directives to metadata as `"true"`:

In [ ]:
c = mk_cell('#| export: utils\n1', metadata=dict(nbdev=dict(export='other', eval='false')))
test_eq(c.directives, {'export':'utils', 'eval':'false'})

d = c.directives
d['eval'] = ''
d['hide'] = ''
c.directives = d
test_eq(c.metadata['nbdev'], {'eval': 'true'})
test_eq(c.source, '#| export: utils\n#| hide\n1')
test_eq(c.directives, {'export':'utils', 'hide':'', 'eval':''})

with expect_fail(TypeError, 'must be str'): mk_cell('1', metadata=dict(nbdev=dict(eval=False))).directives

In [ ]:
#| export
@patch
def directive(self:NbCell, name, default=None):
    "Value of directive `name` (`''` if bare), or `default` if absent"
    return self.directives.get(name, default)

@patch
def has_directive(self:NbCell, name): return self.directive(name) is not None

@patch
def remove_directives(self:NbCell, quarto=False):
    "Strip directives from source, keeping cell magics; with `quarto`, instead materialize every directive (metadata included) as a Quarto option line"
    dirs,code = self._partition()
    keep = [o for o in dirs if _cell_mgc.match(o)]
    if quarto: keep += [_dir_line(k, v, self.lang_, quarto=True) for k,v in self.directives.items()]
    self.set_source(''.join(keep + code))

`directive` answers "what value?" and `has_directive` answers "is it there?": a bare directive's value is `''`, which is falsy, so presence tests need `has_directive`. `remove_directives` serves the two pipeline consumers: the default strips every directive line (module export needs clean source), while `quarto=True` instead rewrites the cell with every directive (comments and metadata alike) as a Quarto option line, bare ones as `name: true`; Quarto consumes the options it knows and quietly turns unknown ones into `data-*` attributes.

In [ ]:
c = mk_cell('%%time\n#| exports: utils\n#| hide\n#| code-fold: show\nslow()', metadata=dict(nbdev=dict(echo='false')))
test_eq(c.directive('exports'), 'utils')
assert c.has_directive('hide') and not c.has_directive('eval')

c.remove_directives(quarto=True)
test_eq(c.source, '%%time\n#| exports: utils\n#| hide: true\n#| code-fold: show\n#| echo: false\nslow()')

c = mk_cell('%%time\n#| exports: utils\n#| hide\n#| code-fold: show\nslow()', metadata=dict(nbdev=dict(echo='false')))
c.remove_directives()
test_eq(c.source, '%%time\nslow()')

## Writing a notebook

In [ ]:
#| export
def _split_mime(data):
    "Copy of mimebundle `data` with multiline text split to lists of lines, like nbformat's `split_lines`"
    return {k: v.splitlines(True) if isinstance(v,str) and (k.startswith('text/') or k in _non_text_split_mimes) else v
            for k,v in data.items()}

def _split_cell(c):
    "Copy of cell `c` with source, attachment, and output text split to lists of lines"
    c = dict(c)
    if isinstance(c.get('source'), str): c['source'] = c['source'].splitlines(True)
    if c.get('attachments'): c['attachments'] = {k:_split_mime(v) for k,v in c['attachments'].items()}
    if c.get('cell_type')=='code' and c.get('outputs'):
        c['outputs'] = [dict(o, data=_split_mime(o['data'])) if o.get('output_type') in ('execute_result','display_data') and 'data' in o
                        else dict(o, text=o['text'].splitlines(True)) if o.get('output_type')=='stream' and isinstance(o.get('text'),str)
                        else o for o in c['outputs']]
    return c

def nb2dict(d, k=None):
    "Convert parsed notebook to `dict`"
    if k=='cells' and isinstance(d, list): d = list(map(_split_cell, d))
    if isinstance(d, list): return list(map(nb2dict,d))
    if not isinstance(d, dict): return d
    return dict(**{k:nb2dict(v,k) for k,v in d.items() if k[-1] != '_'})


This returns the exact same dict as is read from the notebook JSON.

In [ ]:
minimal_fn = Path('../tests/minimal.ipynb')
minimal = read_nb(minimal_fn)
minimal_dict = _read_json(minimal_fn)
assert minimal_dict==nb2dict(minimal)

In [ ]:
#| hide
# caches and lang stamp are working state, not notebook content: they never serialize
nb = new_nb([mk_cell('#| hide\n1')])
nb.cells[0].directives
assert not any(k[-1]=='_' for k in nb2dict(nb)['cells'][0])

In [ ]:
#| export
def nb2str(nb):
    "Convert `nb` to a `str`"
    # Unconditional: a plain dict can hold live NbCells (e.g. spliced in by a processor), whose working
    # attrs (`idx_` etc) and str `source` must not serialize. Idempotent on already-canonical dicts.
    nb = nb2dict(nb)
    return dumps(nb, sort_keys=True, indent=1, ensure_ascii=False) + "\n"

To save a notebook we first need to convert it to a `str`:

In [ ]:
print(nb2str(minimal)[:45])

{
 "cells": [
  {
   "cell_type": "markdown",


In [ ]:
#| export
def write_nb(nb, path):
    "Write `nb` to `path`"
    new = nb2str(nb)
    path = Path(path)
    old = Path(path).read_text(encoding='utf-8') if path.exists() else None
    if new!=old:
        with open(path, 'w', encoding='utf-8') as f: f.write(new)

This returns the exact same string as saved by Jupyter.

In [ ]:
tmp = Path('tmp.ipynb')
try:
    minimal_txt = minimal_fn.read_text()
    write_nb(minimal, tmp)
    test_eq(minimal_txt, tmp.read_text())
finally: tmp.unlink()

## Cell tools

#| export
Cell tools apply `fastcore.tools`' string editing primitives to one notebook cell's source, addressed by path and cell id, mirroring that module's file tools: the same operations and parameters, with `path, cell_id` in place of `path`. Each editor returns a diff of the change, and `view_cell` shows a cell's source with optional line numbers or exhash addresses.

Naming and parameter conventions shared across the editing toolkit are documented in `fastcore.editskill`, which also re-exports this module's editing tools.

In [ ]:
#| export
_cell_edit_doc = """
This is a *cell* editing function.

Cell editing standard parameters are `path`: notebook file to modify, and `cell_id`: id of the cell to edit (exact, or unique prefix)

returns: diff of changes, or "none: No changes.", or "error: ..."
"""

def _nb_cell(nb, cell_id):
    "Cell in `nb` with id `cell_id` (exact match, or unique prefix)"
    res = [c for c in nb.cells if c.id==cell_id] or [c for c in nb.cells if c.id.startswith(cell_id)]
    if len(res)!=1: raise KeyError(f"{'ambiguous' if res else 'no'} cell id: {cell_id!r}")
    return res[0]

def cell_edit(f, name=None):
    def wrapper(path:str, cell_id:str, *args, **kw):
        nb = read_nb(path)
        cell = _nb_cell(nb, cell_id)
        text = cell.source
        try: new_text = f(text, *args, **kw)
        except ValueError as e: return PrettyString(f'error: {e}')
        cell.source = new_text
        write_nb(nb, path)
        return PrettyString(str_diff(text, new_text) or 'none: No changes.')
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _cell_edit_doc
    return res

In [ ]:
#| export
cell_insert_line = cell_edit(insert_line, 'cell_insert_line')
cell_str_replace = cell_edit(str_replace, 'cell_str_replace')
cell_strs_replace = cell_edit(strs_replace, 'cell_strs_replace')
cell_replace_lines = cell_edit(replace_lines, 'cell_replace_lines')
cell_del_lines = cell_edit(del_lines, 'cell_del_lines')

In [ ]:
#| export
def view_cell(
    path:str, # Notebook file to read
    cell_id:str, # Id of the cell to view (exact, or unique prefix)
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; may be past EOF, which clamps to the last line)
    nums:bool=True, # Show line numbers?
    lnhashs:bool=False # Show exhash `lineno|hash|` addresses instead of line numbers?
):
    "View a cell's source, optionally limited to 1-based line range"
    lines = _nb_cell(read_nb(path), cell_id).source.splitlines()
    if not lines: return ''
    if end_line is None or end_line > len(lines): end_line = len(lines)
    if end_line < 0: end_line = len(lines)+end_line+1
    if not (1 <= start_line <= len(lines)): return f'error: Invalid start_line {start_line}. Valid range: 1-{len(lines)}'
    fmt = (lambda i,l: lnhash(i,l)+l) if lnhashs else (lambda i,l: f'{i}: {l}') if nums else (lambda i,l: l)
    return PrettyString('\n'.join(fmt(i,l) for i,l in enumerate(lines[start_line-1:end_line], start_line)))

A round-trip over a temporary notebook exercises each editor and view mode:

In [ ]:
tmp_nb = Path('tmp_cells.ipynb')
write_nb(new_nb([mk_cell('a=1\nprint(a)'), mk_cell('# title', 'markdown')]), tmp_nb)
cid = read_nb(tmp_nb).cells[0].id
test_eq(str(view_cell(tmp_nb, cid)), '1: a=1\n2: print(a)')
test_eq(str(view_cell(tmp_nb, cid, start_line=2, nums=False)), 'print(a)')
test_eq(str(view_cell(tmp_nb, cid, lnhashs=True)).splitlines()[0], lnhash(1,'a=1')+'a=1')
res = cell_str_replace(tmp_nb, cid, 'a=1', 'a=2')
assert '-a=1' in str(res) and '+a=2' in str(res)
test_eq(read_nb(tmp_nb).cells[0].source, 'a=2\nprint(a)')
cell_insert_line(tmp_nb, cid, 0, 'import sys')
cell_del_lines(tmp_nb, cid, 1, 1)
test_eq(read_nb(tmp_nb).cells[0].source, 'a=2\nprint(a)')
cell_replace_lines(tmp_nb, cid, new_content='b=3')
test_eq(read_nb(tmp_nb).cells[0].source, 'b=3\n')  # replace_lines yields a line block, so a trailing newline
assert str(cell_str_replace(tmp_nb, cid, 'q', 'r')).startswith('error:')
with expect_fail(KeyError, 'no cell id'): view_cell(tmp_nb, 'zzzz')
tmp_nb.unlink()

Here's how to put all the pieces of `fastcore.nbio` together:

In [ ]:
nb = new_nb([mk_cell('print(1)')])
path = Path('test.ipynb')
write_nb(nb, path)
nb2 = read_nb(path)
print(nb2.cells)
path.unlink()

[{'cell_type': 'code', 'execution_count': None, 'id': 'b00e7cbc', 'metadata': {}, 'outputs': [], 'source': 'print(1)', 'idx_': 0, 'lang_': 'python'}]


Notebook files on disk store multiline text as lists of lines (nbformat's `split_lines`), for source, stream `text`, textual mime `data`, and attachments. Like nbformat's own reader, we join these to plain strings on read, and split them back on write, so in-memory code always sees strings while files round-trip byte-identically with Jupyter's. JSON mimes and error tracebacks are genuinely structured, so they pass through untouched.

In [ ]:
disk = dict(nbformat=4, nbformat_minor=5, metadata={}, cells=[
    dict(cell_type='code', id='c0', metadata={}, execution_count=1, source=['a=1\n','a'],
         outputs=[dict(output_type='execute_result', metadata={}, execution_count=1,
                       data={'text/plain':['hi\n','there'], 'text/markdown':['single'],
                             'application/json':{'a':[1,2]}, 'image/png':'iVBOR\nw0KG=='}),
                  dict(output_type='stream', name='stdout', text=['s1\n','s2\n']),
                  dict(output_type='error', ename='E', evalue='e', traceback=['t1','t2'])]),
    dict(cell_type='markdown', id='m0', metadata={}, source=['# t\n','x'],
         attachments={'im.txt':{'text/plain':['a\n','b']}, 'im.png':{'image/png':'aGk='}})])
tmp = Path('tmp.ipynb')
try:
    tmp.write_text(dumps(disk))
    nb = read_nb(tmp)
    res,strm,err = nb.cells[0].outputs
    test_eq(res['data']['text/plain'], 'hi\nthere')        # textual data joined on read
    test_eq(res['data']['text/markdown'], 'single')        # one-element lists too
    test_eq(res['data']['application/json'], {'a':[1,2]})  # JSON mimes untouched
    test_eq(res['data']['image/png'], 'iVBOR\nw0KG==')
    test_eq(strm['text'], 's1\ns2\n')                      # stream text joined
    test_eq(err['traceback'], ['t1','t2'])                 # tracebacks stay lists
    test_eq(nb.cells[1].attachments['im.txt']['text/plain'], 'a\nb')
    test_eq(nb2dict(nb), disk)                             # splitting on save restores the disk form exactly
finally: tmp.unlink()

## Validation

Direct edits to cell dicts can produce notebooks that Jupyter and other tools reject: an `outputs` key on a markdown cell, a code cell missing `execution_count`, duplicate ids. `validate_cell` and `validate_nb` are cheap structural checks for exactly those mistakes. They raise `ValueError` naming the offending cell, and never repair: fixing is the caller's decision. This is not the full nbformat schema, just the rules whose violation breaks notebooks in practice.

In [ ]:
#| export
def validate_cell(cell, idx=None):
    "Raise `ValueError` for structural problems in notebook cell dict `cell`; returns it unchanged if fine"
    where = f"cell {cell.get('id', idx)}"
    ct = cell.get('cell_type')
    if ct not in ('code','markdown','raw'): raise ValueError(f"{where}: unknown cell_type {ct!r}")
    src = cell.get('source', '')
    if not (isinstance(src,str) or (isinstance(src,list) and all(isinstance(o,str) for o in src))):
        raise ValueError(f"{where}: source must be str or list of str")
    if not isinstance(cell.get('metadata', {}), dict): raise ValueError(f"{where}: metadata must be a dict")
    if ct=='code':
        if not isinstance(cell.get('outputs'), list): raise ValueError(f"{where}: code cell requires an outputs list")
        if 'execution_count' not in cell: raise ValueError(f"{where}: code cell requires execution_count")
    elif k := first(k for k in ('outputs','execution_count') if k in cell): raise ValueError(f"{where}: {k} not allowed in a {ct} cell")
    return cell

def validate_nb(nb):
    "Raise `ValueError` for structural problems in notebook `nb`; returns it unchanged if fine"
    if k := first(k for k in ('cells','metadata','nbformat') if k not in nb): raise ValueError(f"notebook requires {k!r}")
    for i,c in enumerate(nb['cells']): validate_cell(c, i)
    ids = [c.get('id') for c in nb['cells'] if c.get('id')]
    if dups := {o for o in ids if ids.count(o)>1}: raise ValueError(f"duplicate cell id(s): {', '.join(sorted(dups))}")
    return nb

A valid notebook passes through unchanged, and each rule fails loudly, naming the cell (the markdown-with-outputs case is a real one: stray keys from hand-edited files):

In [ ]:
vnb = new_nb([mk_cell('1+1'), mk_cell('a note', 'markdown')])
test_eq(validate_nb(vnb), vnb)
vnb.cells[1]['outputs'] = []
with expect_fail(ValueError, 'not allowed in a markdown cell'): validate_nb(vnb)
del vnb.cells[1]['outputs'], vnb.cells[0]['execution_count']
with expect_fail(ValueError, 'requires execution_count'): validate_nb(vnb)

In [ ]:
dup = new_nb([mk_cell('a', id='x1'), mk_cell('b', id='x1')])
with expect_fail(ValueError, 'duplicate cell id'): validate_nb(dup)
with expect_fail(ValueError, 'unknown cell_type'): validate_cell(dict(cell_type='wat', source=''))
with expect_fail(ValueError, 'str or list of str'): validate_cell(dict(cell_type='raw', source=[1,2]))

In [ ]:
#| export
def repair_cell(cell, idx=None):
    "Fix deterministic structural problems in `cell`, returning a list of repairs made"
    res,where = [],f"cell {cell.get('id', idx)}"
    src = cell.get('source', '')
    if not (isinstance(src,str) or (isinstance(src,list) and all(isinstance(o,str) for o in src))):
        cell['source'] = ''.join(map(str, listify(src)))
        res.append(f"{where}: coerced source to text")
    if not isinstance(cell.get('metadata', {}), dict):
        cell['metadata'] = {}
        res.append(f"{where}: reset non-dict metadata")
    if cell.get('cell_type')=='code':
        if not isinstance(cell.get('outputs'), list):
            cell['outputs'] = []
            res.append(f"{where}: added outputs list")
        if 'execution_count' not in cell:
            cell['execution_count'] = None
            res.append(f"{where}: added execution_count")
    else:
        for k in ('outputs','execution_count'):
            if k in cell:
                del cell[k]
                res.append(f"{where}: removed {k}")
    return res

def repair_nb(nb):
    "Fix deterministic structural problems in `nb`, returning a list of repairs made"
    res = []
    for k,v in dict(cells=[], metadata={}, nbformat=4).items():
        if k not in nb:
            nb[k] = v
            res.append(f"added notebook {k}")
    for i,c in enumerate(nb['cells']): res += repair_cell(c, i)
    seen = set()
    for c in nb['cells']:
        if (i := c.get('id')) in seen:
            c['id'] = rtoken_hex(4)
            res.append(f"cell {i}: duplicate id, regenerated as {c['id']}")
        seen.add(c.get('id'))
    return res

Each validation rule has a deterministic repair, applied by `repair_nb`: non-code cells lose stray `outputs`/`execution_count`, code cells gain missing ones, broken `source`/`metadata` are coerced, missing notebook fields are added, and duplicate cell ids are regenerated (the first occurrence keeps the id). Unknown `cell_type`s are left alone, since no repair can know the intent. The returned list says what was done, so callers can report or count repairs; a valid notebook returns `[]` and is untouched.

In [ ]:
bad = dict2nb(dict(cells=[
    dict(cell_type='markdown', source='hi', outputs=[], execution_count=1, id='x1'),
    dict(cell_type='code', source='1+1', id='x1'),
], metadata={}, nbformat=4, nbformat_minor=5))
with expect_fail(ValueError): validate_nb(bad)

repairs = repair_nb(bad)
validate_nb(bad)
test_eq(len(repairs), 5)
assert bad.cells[0].id != bad.cells[1].id
test_eq(repair_nb(bad), [])

## Output rendering

In [ ]:
#| export
from fastcore.xml import NB,to_xml,ft

In [ ]:
#| export
def preferred_out(data, html1st=True, include_imgs=False):
    preftyps = ('application/javascript', 'text/latex')
    preftyps = (('text/html', 'text/markdown') if html1st else ('text/markdown', 'text/html')) + preftyps
    if include_imgs: preftyps += 'image/jpeg','image/png','image/svg+xml'
    preftyps += ('text/plain',)
    for mt in preftyps:
        if (text := data.get(mt)): return mt,text
    return 'text/plain',''

`preferred_out` selects the best MIME type from an output's `data` dict, preferring HTML by default:

In [ ]:
data = dict(text_plain=['42'], **{'text/html': ['<b>42</b>'], 'text/plain': ['42']})
preferred_out(data), preferred_out(data, html1st=False)

(('text/html', ['<b>42</b>']), ('text/html', ['<b>42</b>']))

In [ ]:
#| export
def _join(d): return ''.join(d) if isinstance(d, list) else d

In [ ]:
#| export
def mk_stream(name, text):
    "Helper to create an output stream dict"
    return {'output_type': 'stream', 'name': name, 'text': text}

def _mkout(typ, metadata=None, **data):
    return dict(output_type=typ, data={k.replace('_','/'):v for k,v in data.items()}, metadata=metadata or {})

def mk_result(metadata=None, **data):
    "Helper to create an execute_result output dict"
    return _mkout('execute_result', metadata, **data)

def mk_display(metadata=None, **data):
    "Helper to create a display_data output dict"
    return _mkout('display_data', metadata, **data)

def mk_error(traceback, ename='', evalue=''):
    "Helper to create an error output dict"
    return dict(output_type='error', ename=ename, evalue=evalue, traceback=listify(traceback))

In [ ]:
#| export
def concat_streams(outputs):
    "Concatenate stream outputs by name (stdout/stderr), preserving execute_result at end"
    streams, res, execute_results = {}, [], []
    for out in outputs:
        if out['output_type'] == 'stream':
            name = out['name']
            streams[name] = streams.get(name, '') + _join(out['text'])
        elif out['output_type'] in ('error','execute_result'): execute_results.append(out)
        else: res.append(out)
    for name in ('stdout','stderr'):
        if name in streams: res.append(mk_stream(name, clean_cli_output(streams[name], strip=False)))
    res.extend(execute_results)
    return res

`concat_streams` merges consecutive stream outputs by name and moves execute_results to the end, like standard jupyter output rendering:

In [ ]:
outs = [mk_result(text_plain=['42']),
        mk_stream('stdout', 'hello '), mk_stream('stdout', 'world\n'), mk_stream('stderr', 'warn\n')]
outs

[{'output_type': 'execute_result',
  'data': {'text/plain': ['42']},
  'metadata': {}},
 {'output_type': 'stream', 'name': 'stdout', 'text': 'hello '},
 {'output_type': 'stream', 'name': 'stdout', 'text': 'world\n'},
 {'output_type': 'stream', 'name': 'stderr', 'text': 'warn\n'}]

In [ ]:
concat_streams(outs)

[{'output_type': 'stream', 'name': 'stdout', 'text': 'hello world\n'},
 {'output_type': 'stream', 'name': 'stderr', 'text': 'warn\n'},
 {'output_type': 'execute_result',
  'data': {'text/plain': ['42']},
  'metadata': {}}]

Carriage-return overwrites apply across chunk boundaries, as they would on a live terminal — a progress line ending in `\r` is overwritten by the next chunk, but a final `\r` (nothing follows it) leaves the text visible:

In [ ]:
prog = [mk_stream('stdout', 'step 1\r'), mk_stream('stdout', 'step 2\r'), mk_stream('stdout', 'done\n')]
test_eq(concat_streams(prog), [mk_stream('stdout', 'done\n')])
test_eq(concat_streams([mk_stream('stdout', 'working\r')]), [mk_stream('stdout', 'working')])

In [ ]:
#| export
@delegates(preferred_out)
def preferred_msg_out(out, **kwargs):
    "Preferred mime type and content for any Jupyter output dict (stream, error, or data-bearing)"
    typ = out['output_type']
    if typ == 'stream': return 'text/plain', _join(out.get('text', ""))
    elif typ == 'error': return 'text/plain', '\n'.join(out.get('traceback', []))
    elif typ in ('execute_result', 'display_data'): return preferred_out(out.get('data', {}), **kwargs)
    return 'text/plain',f'Error: Failed to parse unknown output - {out}'

`preferred_msg_out` extends `preferred_out` to any output dict: streams and errors are always plain text, while data-bearing outputs go through mime preference:

In [ ]:
test_eq(preferred_msg_out(mk_stream('stdout', ['a\n','b\n'])), ('text/plain', 'a\nb\n'))
test_eq(preferred_msg_out(mk_result(text_html=['<b>4</b>'], text_plain=['4'])), ('text/html', ['<b>4</b>']))
test_eq(preferred_msg_out(mk_result(text_markdown=['*4*'], text_html=['<b>4</b>']), html1st=False)[0], 'text/markdown')

In [ ]:
#| export
def render_output(out):
    "Convert a single output dict to an HTML string"
    def _fmt(text):
        res = ansi2html(str(text))
        return f'<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">{res}</code></pre>'
    ptyp,d = preferred_msg_out(out, html1st=True, include_imgs=True)
    d = _join(d)
    if   ptyp=='text/plain': return _fmt(d)
    elif ptyp=='text/html': return d
    elif ptyp=='application/javascript': return f'<script>{d}</script>'
    elif ptyp=='text/markdown': return d
    elif ptyp=='text/latex': return f'<div>{d}</div>'
    elif ptyp=='image/jpeg': return f'<img src="data:image/jpeg;base64,{d}"/>'
    elif ptyp=='image/png':  return f'<img src="data:image/png;base64,{d}"/>'
    elif ptyp=='image/svg+xml': return d
    return ''

In [ ]:
print(render_output(mk_result(text_plain=['42'])))

<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">42</code></pre>


In [ ]:
#| export
def render_outputs(outputs):
    "Render a full list of outputs, concatenating streams first."
    if (not isinstance(outputs, (list,tuple))) or (outputs and not isinstance(outputs[0],dict)): return ''
    return '\n'.join(render_output(o) for o in concat_streams(outputs))

In [ ]:
print(render_outputs(outs))

<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">hello world
</code></pre>
<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">warn
</code></pre>
<pre class="!border-0 !rounded-none !my-0 !p-0"><code class="nohighlight">42</code></pre>


In [ ]:
#| export
def _render_text(out, html1st=False):
    typ = out['output_type']
    mime,d = preferred_msg_out(out, html1st=html1st, include_imgs=False)
    d = _join(d)
    if not d: return None
    attrs = {}
    if typ == 'stream': typ = out.get('name')
    elif typ in ('execute_result','display_data') and mime!='text/plain': attrs['mime'] = mime
    body = f'\n{d}' if d.endswith('\n') else f'\n{d}\n'
    return d, to_xml(ft(typ, body, **attrs), do_escape=False, indent=False)

def render_text(outputs, html1st=False):
    "Render notebook outputs to concise text, using XML-ish tags when multiple outputs are present."
    if (not isinstance(outputs, (list,tuple))) or (outputs and not isinstance(outputs[0],dict)): return ''
    items = [o for out in concat_streams(outputs) if (o := _render_text(out, html1st=html1st))]
    if not items: return ''
    return items[0][0] if len(items)==1 else '\n'.join(o[1] for o in items)

A single output renders as plain text directly:...

In [ ]:
print(render_text([outs[0]]))

42


...but multiple outputs get wrapped in XML-ish tags so they stay distinguishable. For rich outputs we prefer markdown by default; pass `html1st=True` to prefer HTML instead. Outputs without a text representation (e.g. images alone) render as empty.

In [ ]:
print(render_text(outs))

<stdout>
hello world
</stdout>
<stderr>
warn
</stderr>
<execute_result>
42
</execute_result>


In [ ]:
disp = [mk_display(text_html=['<b>42</b>'], text_markdown=['**42**'], text_plain=['42'])]
test_eq(render_text(disp), '**42**')
test_eq(render_text(disp, html1st=True), '<b>42</b>')

test_eq(render_text([mk_display(image_png='abc')]), '')
test_eq(render_text([mk_error('oops')]), 'oops')

In [ ]:
nb.cells[0].outputs = [mk_stream('stdout', '1\n')]

## Notebook class

In [ ]:
#| export
def item2xml(
    typ, # Tag name: the cell or message type, e.g. 'code', 'markdown', 'raw', 'prompt'
    content='', # The item's source text
    out='', # Rendered output text
    id=None, # Optional id attribute
    meta=None, # Cell/message metadata: directives in its `nbdev` dict render as attrs, bare ones as bare attrs
    **attrs, # Extra attributes; falsy values are dropped, names stay verbatim (no hyphenation)
):
    "A notebook cell or dialog message as concise XML: content, then an `<out>` section when `out` is non-empty"
    kw = {k:v for k,v in {'id':id, **_dir_attrs(meta), **attrs}.items() if v}
    return ft(typ, content, ft('out', out), attrmap=str, **kw) if out else ft(typ, content, attrmap=str, **kw)

`item2xml` renders notebook cells and dialog messages to LLM-friendly XML (`cell2xml` below and `llmsurgery`'s message renderers build on it). The content sits directly inside the type tag, with no wrapper of its own, and an `<out>` section marks where output begins - so an item without output carries no extra tags at all. Passing `meta` renders the metadata's nbdev directives as attributes, so every projection built on `item2xml` shows them the same way.

In [ ]:
test_eq(to_xml(item2xml('code', 'x*2', '42', id='ab')), '<code id="ab">x*2<out>42</out></code>')
to_xml(item2xml('markdown', '# hi', id='cd'))

'<markdown id="cd"># hi</markdown>'

Falsy attrs are dropped, literal True attrs have no value:

In [ ]:
test_eq(to_xml(item2xml('code', 'x', time='', kind='system', foo=True)), '<code kind="system" foo>x</code>')
test_eq(to_xml(item2xml('code', 'x', meta=dict(nbdev=dict(hide='true', eval='false')))), '<code hide eval="false">x</code>')

In [ ]:
#| export
def cell2xml(cell, ids=True, incl_out=True):
    "Convert NbCell to concise XML format"
    outputs = getattr(cell, 'outputs', None) if incl_out else None
    return item2xml(cell.cell_type, cell.source, render_text(outputs) if outputs else '', id=cell.id if ids else None,
                    meta=cell.get('metadata'))

def cells2xml(cells, wrap=NB, ids=True, incl_out=True, **kw):
    "Convert notebook cells to XML format"
    return to_xml(wrap(*[cell2xml(c, ids=ids, incl_out=incl_out) for c in cells], **kw), do_escape=False)


We can view any notebook as concise XML. For instance, our minimal notebook:

In [ ]:
print(cells2xml(nb.cells, incl_out=False))

<nb><code id="c0">a=1
a</code><markdown id="m0"># t
x</markdown></nb>


In [ ]:
repr(cell2xml(nb.cells[0], incl_out=False))

'<code id="c0">a=1\na</code>'

In [ ]:
repr(cell2xml(nb.cells[0], incl_out=True))

'<code id="c0">a=1\na<out>1\n</out></code>'

In [ ]:
# Metadata directives render as attrs: bare as bare, others verbatim (names unhyphenated)
c = mk_cell('1+1', metadata=dict(nbdev=dict(hide='true', default_exp='core')))
test_eq(repr(cell2xml(c, incl_out=False)), f'<code id="{c.id}" hide default_exp="core">1+1</code>')

In [ ]:
#| export
class Notebook:
    "Read, query, and edit Jupyter notebooks"
    def __init__(self, nb, path=None):
        if not path: path = Path("unnamed.ipynb")
        store_attr()

    @property
    def _id2cell(self): return {c.id: c for c in self.nb.cells}

    @classmethod
    def open(cls, path):
        path = Path(path).resolve()
        return cls(read_nb(path), path)

    def save(self, path=None): write_nb(self.nb, path or self.path)

    @property
    def cells(self): return self.nb.cells
    @property
    def meta(self): return self.nb.metadata

    def __getitem__(self, k): return self.cells[k] if isinstance(k, (int, slice)) else self._id2cell[k]
    def __setitem__(self, k, source): self[k].set_source(source)
    def __len__(self): return len(self.cells)
    def __iter__(self): return iter(self.cells)
    def __contains__(self, k): return k in self._id2cell
    def __delitem__(self, k): self.cells.remove(self[k])
    def __repr__(self): return cells2xml(self.cells, path=self.path)
    @property
    def concise (self): return cells2xml(self.cells, path=self.path.name, incl_out=False)

We can now open a notebook and access its metadata and cells:

In [ ]:
nbo = Notebook.open(minimal_fn)
list(nbo.meta), len(nbo.cells), len(nbo)

(['solveit_dialog_mode', 'solveit_ver'], 2, 2)

In [ ]:
nbo.path.name

'minimal.ipynb'

In [ ]:
[o.id for o in nbo]

['801558df', 'e2147a69']

In [ ]:
'e2147a69' in nbo, 'nonexistent' in nbo

(True, False)

Notebooks' repr is their xml:

In [ ]:
nbo

<nb path="/Users/jhoward/aai-ws/fastcore/tests/minimal.ipynb"><markdown id="801558df">## A minimal notebook</markdown><code id="e2147a69"># Do some arithmetic
1+1<out>2</out></code></nb>

You can also get a more concise version that doesn't include outputs or the full path:

In [ ]:
print(nbo.concise)

<nb path="minimal.ipynb"><markdown id="801558df">## A minimal notebook</markdown><code id="e2147a69"># Do some arithmetic
1+1</code></nb>


Cells can be accessed by integer index or by their string id:

In [ ]:
nbo[0].source

'## A minimal notebook'

In [ ]:
nbo['e2147a69'].source

'# Do some arithmetic\n1+1'

You can directly set a cell's source by id or index:

In [ ]:
nbo['e2147a69'] = '2+2'
nbo['e2147a69'].source


'2+2'

You can also update outputs and metadata directly on a cell:

In [ ]:
nbo['e2147a69'].outputs = [{'output_type': 'execute_result', 'data': {'text/plain': ['4']}}]
nbo['e2147a69'].outputs

[{'output_type': 'execute_result', 'data': {'text/plain': ['4']}}]

In [ ]:
nbo['e2147a69'].metadata['custom'] = True
nbo['e2147a69'].metadata

<div class="prose" markdown="1">

```python
{'custom': True, 'time_run': '2026-01-04T20:52:49.901559+00:00'}
```

</div>

The `add` method inserts a new cell at a given position (defaulting to the end):

In [ ]:
#| export
@patch
def add(self:Notebook, source, cell_type='code', idx=None, after=None, before=None, **kwargs):
    "Add a new cell with `source` at `idx` (default: end), or `after`/`before` a cell id"
    if after: idx = next((i for i,c in enumerate(self.cells) if c.id==after), None)
    if idx is None and after: raise KeyError(after)
    if idx is not None: idx += 1
    elif before: idx = next((i for i,c in enumerate(self.cells) if c.id==before), None)
    if idx is None and before: raise KeyError(before)
    c = mk_cell(source, cell_type=cell_type, **kwargs)
    if idx is None: self.cells.append(c)
    else: self.cells.insert(idx, c)
    return c

In [ ]:
nbo.add('print("hello")')
nbo.add('# A heading', cell_type='markdown', idx=0)
len(nbo), nbo[0].source

(4, '## A minimal notebook')

Cells can also be inserted relative to an existing cell by id:

In [ ]:
cid = nbo[0].id
nbo.add('# After first', cell_type='markdown', after=cid)
nbo.add('# Before first', cell_type='markdown', before=cid)
[c.source for c in nbo[:3]]

['# Before first', '## A minimal notebook', '# After first']

In [ ]:
#| export
@patch
def md(self:Notebook, source, idx=None, after=None, before=None, **kwargs):
    "Add a new cell with `source` at `idx` (default: end), or `after`/`before` a cell id"
    return self.add(source, cell_type='markdown', idx=idx, after=after, before=before, **kwargs)

`md` is a shortcut to `add(..., cell_type='markdown')`

In [ ]:
nbo.md('A note')
len(nbo), nbo[-1].cell_type

(7, 'markdown')

You can delete by id or index:

In [ ]:
prev_len = len(nbo)
del nbo[0]
len(nbo) == prev_len - 1


True

The `find` method searches cell sources by regex, returning matching cells:

In [ ]:
#| export
@patch
def find(self:Notebook, pat, cell_type=None):
    "Find cells with source matching regex `pat`"
    return [c for c in self.cells if re.search(pat, c.source) and (not cell_type or c.cell_type==cell_type)]

In [ ]:
nbo.find(r'\d\+\d', cell_type='code')

[{'cell_type': 'code',
  'execution_count': None,
  'id': 'e2147a69',
  'metadata': {'time_run': '2026-01-04T20:52:49.901559+00:00', 'custom': True},
  'outputs': [{'output_type': 'execute_result',
    'data': {'text/plain': ['4']}}],
  'source': '2+2',
  'idx_': 1,
  'lang_': 'python'}]

In [ ]:
#| export
@patch
def move(self:Notebook, src_ids, after=None, before=None):
    "Move cells with `src_ids` after/before a cell id, or to end"
    cells = [self[k] for k in listify(src_ids)]
    for c in cells: self.cells.remove(c)
    if after: idx = next((i+1 for i,c in enumerate(self.cells) if c.id==after), None)
    elif before: idx = next((i for i,c in enumerate(self.cells) if c.id==before), None)
    else: idx = len(self.cells)
    if idx is None: raise KeyError(after or before)
    for i,c in enumerate(cells): self.cells.insert(idx+i, c)

Cells can be moved by id, either relative to another cell or to the end:

In [ ]:
nbo = Notebook.open(minimal_fn)
c0,c1 = nbo[0].id,nbo[1].id
nbo.move(c1, before=c0)
[c.id for c in nbo] == [c1, c0]

True

Use `save` to write to disk:
```py
nbo.save('path.ipynb')
```
If no path is passed, the path used in `open()` will be re-used.

In [ ]:
#| export
@patch
def view(self:Notebook, id, nums=True):
    "Show cell source with optional line numbers"
    lines = self[id].source.splitlines()
    if nums: lines = [f'{i+1:6d} │ {l}' for i,l in enumerate(lines)]
    return '\n'.join(lines)

The `view` method displays a cell's source with optional line numbers:

In [ ]:
print(nbo.view('e2147a69'))

     1 │ # Do some arithmetic
     2 │ 1+1


## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()